# 18a — Associative Memory

Corresponds to: Note 18 — Associative Memory

**Libraries introduced in this notebook:** `numpy` — already introduced in Note 03. No new libraries.

**What this notebook demonstrates:**
1. Hebbian outer product storage rule for auto-associative memory
2. Asynchronous retrieval from a noisy probe — verifies Part 3 worked example
3. Hetero-associative storage and single-pass retrieval — verifies Part 5 worked example
4. Capacity experiment: retrieval success rate vs P/N ratio

## Part 1: Hebbian Storage — Auto-Associative

Store two patterns of length N = 4 using the outer product rule:
W = (1/N) * sum of x^m (x^m)^T, diagonal zeroed.
This reproduces the worked example from Note 18, Part 2.

In [ ]:
import numpy as np

N = 4
x1 = np.array([ 1,  1, -1,  1])
x2 = np.array([-1,  1,  1,  1])

print('x1:', x1)
print('x2:', x2)
print('x1 . x2 =', x1 @ x2, '(orthogonal)')

op1 = np.outer(x1, x1)
op2 = np.outer(x2, x2)
print('\nOuter product x1(x1)^T:')
print(op1)
print('\nOuter product x2(x2)^T:')
print(op2)

W = (1 / N) * (op1 + op2)
np.fill_diagonal(W, 0)
print('\nW = (1/4)(sum), diagonal zeroed:')
print(W)

Verify that both stored patterns are fixed points of the update rule: sign(W x^m) == x^m.

In [ ]:
for xm, name in [(x1, 'x1'), (x2, 'x2')]:
    net = W @ xm
    retrieved = np.sign(net)
    print(f'net({name}) = {np.round(net, 4)}')
    print(f'sign(net)  = {retrieved.astype(int)}')
    print(f'Expected   = {xm}')
    print(f'Fixed point: {np.array_equal(retrieved, xm)}')
    print()

## Part 2: Asynchronous Retrieval from a Noisy Probe

Present a probe with s1 flipped from x1. Run asynchronous update (neurons 1 to 4 in order).
This reproduces the worked example from Note 18, Part 3.

In [ ]:
probe = np.array([-1, 1, -1, 1], dtype=float)  # x1 with s1 flipped
print('Probe (x1 with s1 flipped):', probe.astype(int))
print('x1:                        ', x1)
print()

s = probe.copy()
print('Asynchronous update:')
for i in range(N):
    net_i = W[i] @ s
    new_si = 1.0 if net_i >= 0 else -1.0
    print(f'  Neuron {i+1}: net = {net_i:.4f},  s{i+1}: {int(s[i])} -> {int(new_si)}')
    s[i] = new_si

print()
print('After pass 1:', s.astype(int))
print('Matches x1: ', np.array_equal(s, x1))

## Part 3: Hetero-Associative Storage and Retrieval

Store two input-output pairs using W = (1/N) * sum of y^m (x^m)^T.
Retrieval is a single forward pass: y = sign(W x).
This reproduces the worked example from Note 18, Part 5.

In [ ]:
xa = np.array([ 1,  1, -1,  1])
xb = np.array([-1,  1,  1,  1])
ya = np.array([ 1, -1,  1, -1])
yb = np.array([-1, -1,  1,  1])

print('Stored pairs:')
print(f'  xa = {xa}  ->  ya = {ya}')
print(f'  xb = {xb}  ->  yb = {yb}')

W_het = (1 / N) * (np.outer(ya, xa) + np.outer(yb, xb))
print('\nW_het = (1/4)(ya(xa)^T + yb(xb)^T):')
print(W_het)

In [ ]:
for x_in, y_exp, name in [(xa, ya, 'xa'), (xb, yb, 'xb')]:
    net = W_het @ x_in
    y_out = np.sign(net)
    print(f'Input {name}: {x_in}')
    print(f'  net    = {np.round(net, 4)}')
    print(f'  output = {y_out.astype(int)}')
    print(f'  expected {"ya" if name == "xa" else "yb"} = {y_exp}')
    print(f'  match: {np.array_equal(y_out, y_exp)}')
    print()

## Part 4: Capacity Experiment

Sweep P from 1 to 30 for N = 100. For each P, store P random bipolar patterns,
present a probe with one bit flipped, run asynchronous update, and record whether
the correct pattern is retrieved. Average over 200 trials per P value.

In [ ]:
np.random.seed(42)

N_cap = 100
P_values = list(range(1, 31))
n_trials = 200

success_rates = []

for P in P_values:
    successes = 0
    for _ in range(n_trials):
        patterns = np.sign(np.random.randn(P, N_cap))
        patterns[patterns == 0] = 1

        W_cap = np.zeros((N_cap, N_cap))
        for m in range(P):
            W_cap += np.outer(patterns[m], patterns[m])
        W_cap /= N_cap
        np.fill_diagonal(W_cap, 0)

        probe = patterns[0].copy()
        flip_idx = np.random.randint(N_cap)
        probe[flip_idx] *= -1

        s = probe.copy()
        for _ in range(50):
            changed = False
            for i in np.random.permutation(N_cap):
                net_i = W_cap[i] @ s
                new_si = 1.0 if net_i >= 0 else -1.0
                if new_si != s[i]:
                    s[i] = new_si
                    changed = True
            if not changed:
                break

        if np.array_equal(s, patterns[0]):
            successes += 1

    success_rates.append(successes / n_trials)

print('P/N ratio  |  Success rate')
print('-' * 30)
for P, sr in zip(P_values, success_rates):
    bar = '#' * int(sr * 20)
    print(f'  {P/N_cap:.2f}     |  {sr:.2f}  {bar}')

The success rate drops sharply near P/N = 0.14, confirming the theoretical capacity limit.

## Summary

This notebook demonstrated:
1. Hebbian outer product storage: W = (1/N) sum x^m (x^m)^T, diagonal zeroed — stored patterns are fixed points
2. Asynchronous retrieval: one-pass update from a 1-bit-flipped probe recovers x1 exactly (matches Note 18 Part 3)
3. Hetero-associative storage: W = (1/N) sum y^m (x^m)^T — single forward pass y = sign(Wx) retrieves both pairs (matches Note 18 Part 5)
4. Capacity experiment: retrieval success rate collapses near P/N = 0.14, confirming the 0.14N capacity limit